In [3]:
import pandas as pd
icd10_map_dx = pd.read_csv("CCS/DXCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd10_map_pr = pd.read_csv("CCS/PRCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd9_map_dx = pd.read_csv("CCS/ccs_multi_dx_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd9_map_pr = pd.read_csv("CCS/ccs_multi_pr_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)

/tmp/ipykernel_347485/592224789.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd10_map_dx = pd.read_csv("CCS/DXCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_347485/592224789.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd10_map_pr = pd.read_csv("CCS/PRCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_347485/592224789.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd9_map_dx = pd.read_csv("CCS/ccs_multi_dx_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_347485/592224789.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.m

In [12]:
icd9_map_pr

,ICD-9-CM CODE,CCS LVL 1,CCS LVL 1 LABEL,CCS LVL 2,CCS LVL 2 LABEL,CCS LVL 3,CCS LVL 3 LABEL
0,0121,1,Operations on the nervous system,1.1,Incision and excision of CNS [1.],1.1.1,Craniotomy and craniectomy
1,0122,1,Operations on the nervous system,1.1,Incision and excision of CNS [1.],1.1.1,Craniotomy and craniectomy
2,0123,1,Operations on the nervous system,1.1,Incision and excision of CNS [1.],1.1.1,Craniotomy and craniectomy
3,0124,1,Operations on the nervous system,1.1,Incision and excision of CNS [1.],1.1.1,Craniotomy and craniectomy
4,0125,1,Operations on the nervous system,1.1,Incision and excision of CNS [1.],1.1.1,Craniotomy and craniectomy
...,...,...,...,...,...,...,...
3943,9988,16,Miscellaneous diagnostic and therapeutic proce...,16.42,Other therapeutic procedures [231.],16.42.4,Other therapies
3944,9991,16,Miscellaneous diagnostic and therapeutic proce...,16.42,Other therapeutic procedures [231.],16.42.4,Other therapies
3945,9992,16,Miscellaneous diagnostic and therapeutic proce...,16.42,Other therapeutic procedures [231.],16.42.4,Other therapies
3946,9998,16,Miscellaneous diagnostic and therapeutic proce...,16.42,Other therapeutic procedures [231.],16.42.4,Other therapies


In [24]:
import json
from tqdm.notebook import tqdm
with open("mimic-iv_asplenic_4133.json", "r") as jsonfile: 
    data = json.load(jsonfile)
for p in tqdm(data):
    for f,v in p.items():
        if f == "events":
            for e in v:
                if e["type"] == "disease" and e['code_type'] == "ICD-10":
                    try:
                        e["ccs"] = icd10_map_pr[icd10_map_pr["ICD-10-CM CODE"] == e['code']]["CCSR CATEGORY 1 DESCRIPTION"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e["type"] == "therapy" and e['code_type'] == "ICD-10":
                    try:
                        e["ccs"] = icd10_map_pr[icd10_map_pr["ICD-10-PCS"] == e['code']]["PRCCSR DESCRIPTION"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e["type"] == "disease" and e['code_type'] == "ICD-9":
                    try:
                        e["ccs"] = icd9_map_dx[icd9_map_dx["ICD-9-CM CODE"] == e['code']]["CCS LVL 1 LABEL"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e["type"] == "therapy" and e['code_type'] == "ICD-9":
                    try:
                        e["ccs"] = icd9_map_pr[icd9_map_pr["ICD-9-CM CODE"] == e['code']]["CCS LVL 1 LABEL"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e['code_type'] == "PRODUCT-CODE":
                        e["ccs"] = "UNKN"
                else:
                    raise Exception("Wrog code type!")

  0%|          | 0/4133 [00:00<?, ?it/s]

In [25]:
with open("data/mimic-iv_asplenic_4133_ccs.json", "w") as jsonfile: 
    json.dump(data, jsonfile, indent=4)

In [84]:
len(set(icd10_map_dx["ICD-10-CM CODE"]).intersection(icds10_dx)) + len(set(icd10_map_pr["ICD-10-PCS"]).intersection(icds10_dx))

4716

In [86]:
len(set(icd9_map_dx["ICD-9-CM CODE"]).intersection(icds9_dx)) + len(set(icd9_map_pr["ICD-9-CM CODE"]).intersection(icds9_dx))

3944

In [91]:
def find_prefix_matches(code, ccs_codes):
    return [c for c in ccs_codes if c.startswith(code)]
def is_prefix(code, ccs_codes):
    return any(c.startswith(code) for c in ccs_codes)
find_prefix_matches('4476',icd9_map_dx["ICD-9-CM CODE"].values)

['4476']

In [89]:
(icds9_dx - set(icd9_map_dx["ICD-9-CM CODE"])) - set(icd9_map_pr["ICD-9-CM CODE"])

set()

In [88]:
(icds10_dx - set(icd10_map_dx["ICD-10-CM CODE"]))- set(icd10_map_pr["ICD-10-PCS"])

set()

In [90]:
len(icdspc - set(icd10_map_pr["ICD-10-PCS"])), len(set(icd9_map_pr["ICD-9-CM CODE"]).intersection(icdspc))

(2127, 0)